In [36]:
import pandas as pd

# Load the dataset
df = pd.read_csv('jigsaw.csv', encoding='utf-8')

# Check columns and first few rows
print(df.columns)
print(df.head())


Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  


In [38]:
# Prepare data for binary toxicity classification
final_data = df[['comment_text', 'toxic']].rename(columns={'toxic': 'label'})

# Check class distribution
print(final_data['label'].value_counts())


label
0    144277
1     15294
Name: count, dtype: int64


In [39]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    final_data, test_size=0.2, random_state=42, stratify=final_data['label']
)

print("Train class distribution:\n", train_data['label'].value_counts())
print("Validation class distribution:\n", val_data['label'].value_counts())


Train class distribution:
 label
0    115421
1     12235
Name: count, dtype: int64
Validation class distribution:
 label
0    28856
1     3059
Name: count, dtype: int64


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_data['comment_text'])
X_val = vectorizer.transform(val_data['comment_text'])

y_train = train_data['label']
y_val = val_data['label']


In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred, zero_division=0))


              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28856
           1       0.90      0.63      0.74      3059

    accuracy                           0.96     31915
   macro avg       0.93      0.81      0.86     31915
weighted avg       0.96      0.96      0.95     31915



In [ ]:
def get_reward(text):
    X = vectorizer.transform([text])
    pred = clf.predict(X)
    return 1 if pred == 0 else -1  # 1: non-toxic, -1: toxic


sample_texts = [
    "You are so amazing!",
    "I love you",
    "This is a sexy day",
    "You are a fucking idiot"
]

rewards = [(text, get_reward(text)) for text in sample_texts]
print(rewards)


[('You are so amazing!', 1), ('I love you', 1), ('This is a sexy day', 1), ('You are a fucking idiot', -1)]
